In [7]:
!pip install langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------- ----------------------------- 262.1/981.5 kB ? eta -:--:--
     ------------------------------ ------- 786.4/981.5 kB 3.0 MB/s eta 0:00:01
     -------------------------------------- 981.5/981.5 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993251 sha256=a4a804d548a37c3e94b08650bc0143a001f405fcd4aadc59a818598ec3aad10b
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\c1\67\88\e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [9]:
import re
import time
from datetime import datetime

import pandas as pd
from tqdm import tqdm
from langdetect import detect, DetectorFactory
from google_play_scraper import Sort, reviews as gp_reviews

DetectorFactory.seed = 42

APPS = [
    ("Spotify", "com.spotify.music"),
    ("YouTube Music", "com.google.android.apps.youtube.music"),
    ("JioSaavn", "com.jio.media.jiobeats"),
    ("Gaana", "com.gaana"),
    ("Wynk Music", "com.bsbportal.music"),
    # add more here...
]

def clean_text_basic(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def safe_detect_lang(text: str):
    text = (text or "").strip()
    if len(text) < 15:
        return None
    try:
        return detect(text)
    except:
        return None

def fetch_google_play_reviews(app_name, package_name, count=500, lang="en", country="in", sleep_s=0.2):
    all_rows = []
    continuation_token = None
    fetched = 0

    pbar = tqdm(total=count, desc=f"{app_name}", unit="reviews")

    while fetched < count:
        batch_count = min(200, count - fetched)
        result, continuation_token = gp_reviews(
            package_name,
            lang=lang,
            country=country,
            sort=Sort.NEWEST,
            count=batch_count,
            continuation_token=continuation_token,
        )
        if not result:
            break

        for r in result:
            text = clean_text_basic(r.get("content", ""))

            all_rows.append({
                "platform": "google_play",
                "app_name": app_name,
                "rating": float(r.get("score")) if r.get("score") is not None else None,
                "review_text": text,

                # useful dashboard + NLP columns
                "review_id": r.get("reviewId"),
                "review_date": r.get("at").isoformat() if r.get("at") else None,
                "app_version": r.get("reviewCreatedVersion"),
                "thumbs_up": r.get("thumbsUpCount"),
                "user_name": r.get("userName"),
                "country": country,
                "language": lang,
                "detected_lang": safe_detect_lang(text),
                "length_chars": len(text),
                "length_words": len(text.split()) if text else 0,
            })

        fetched += len(result)
        pbar.update(len(result))

        if continuation_token is None:
            break

        time.sleep(sleep_s)

    pbar.close()
    return pd.DataFrame(all_rows)

def main(total_per_app=200, lang="en", country="in", output_csv="music_app_reviews.csv"):
    dfs = []
    for app_name, package in APPS:
        try:
            dfs.append(fetch_google_play_reviews(app_name, package, total_per_app, lang, country))
        except Exception as e:
            print(f"[WARN] Failed for {app_name}: {e}")

    final = pd.concat(dfs, ignore_index=True)

    # NLP-friendly cleaned text
    final["clean_text"] = final["review_text"].astype(str).str.lower()
    final["clean_text"] = final["clean_text"].str.replace(r"http\S+", "", regex=True)
    final["clean_text"] = final["clean_text"].str.replace(r"[^a-z0-9\s]", " ", regex=True)
    final["clean_text"] = final["clean_text"].str.replace(r"\s+", " ", regex=True).str.strip()

    final["extracted_at"] = datetime.utcnow().isoformat()

    # reorder core columns first
    core = ["app_name", "platform", "rating", "review_text"]
    final = final[core + [c for c in final.columns if c not in core]]

    final.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"Saved: {output_csv} | rows={len(final)}")

if __name__ == "__main__":
    main(total_per_app=200)


Wynk Music: 100%|██████████████████████████████████████████████████████████████| 200/200 [00:01<00:00, 116.56reviews/s]

Saved: music_app_reviews.csv | rows=1000



C:\Users\User\AppData\Local\Temp\ipykernel_13744\3087131334.py:106: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  final["extracted_at"] = datetime.utcnow().isoformat()
